# DATA 266 HW1 Neural Networks

In [ ]:
import os
import random
import numpy as np
import torch
import tensorflow as tf

SID4 = 9486
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
tf.random.set_seed(SEED)

print(f"SID4  = {SID4}")
print(f"SEED  = {SEED}")
print(f"SLICE = {SLICE}")
print(f"HP_ID = {HP_ID}")
print(f"CLS_A = {CLS_A}")
print(f"CLS_B = {CLS_B}")


In [ ]:
BASELINE_CONFIG = {
    "hidden_layers": [64, 32],
    "learning_rate": 0.001,
    "epochs": 30
}

MODIFIED_CONFIG = {
    "hidden_layers": [32],
    "learning_rate": 0.001,
    "epochs": 30
}

TRAINING_SEEDS = [SEED, SEED + 1, SEED + 2]

print("Baseline configuration:", BASELINE_CONFIG)
print("Modified configuration:", MODIFIED_CONFIG)
print("Training seeds:", TRAINING_SEEDS)


## 2. Diabetes Dataset

This section loads and inspects the diabetes dataset before preprocessing. No values are modified during this initial inspection.

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

DATA_PATH = Path("data/diabetes.csv")
EXPECTED_SHA256 = "89e33de71cd5afdcd4fe8a722b60c4d69eb8f9b96446cea9347d735ff2df395e"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}")

# The supplied CSV is headerless, so preserve its first row as data.
df = pd.read_csv(DATA_PATH, header=None)
RAW_SHAPE = df.shape
COLUMN_NAMES = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "SourceLabel",
]
df.columns = COLUMN_NAMES

print("Dataset path:", DATA_PATH)
print("Dataset shape:", RAW_SHAPE)
print("Column names:", list(df.columns))
display(df.head())

In [ ]:
print("DataFrame information:")
df.info()

print("Descriptive statistics:")
display(df.describe().T)

In [ ]:
print("Missing-value counts:")
display(df.isna().sum().rename("missing_values"))

print("Duplicate-row count:", int(df.duplicated().sum()))

print("Zero-value counts for numeric columns:")
display((df.select_dtypes(include="number") == 0).sum().rename("zero_values"))

In [ ]:
# The headerless dataset uses its final column as the binary target.
TARGET_COLUMN = df.columns[-1]
target_values = sorted(df[TARGET_COLUMN].dropna().unique().tolist())
target_counts = df[TARGET_COLUMN].value_counts().sort_index()
target_percentages = (
    df[TARGET_COLUMN].value_counts(normalize=True).sort_index() * 100
).round(4)

print("Confirmed target column:", TARGET_COLUMN)
print("Target unique values:", target_values)
print("Target class counts:")
display(target_counts.rename("count"))
print("Target class percentages:")
display(target_percentages.rename("percentage"))

### Initial Observations

- The dataset contains 759 rows and 9 source columns, giving 8 input features and 1 source-label column.
- The CSV is headerless, but its matched schema uses the standard order: Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age, and SourceLabel.
- SourceLabel is binary with values `0` and `1`; the assignment-provided encoding is `0 = diabetes` and `1 = no diabetes`. The modeling target will be remapped later as `Diabetes = 1 - SourceLabel`, so `Diabetes=1` means diabetes.
- There are no explicit missing values and no duplicate rows. The source features are already scaled approximately to `[-1, 1]`.
- Zero is valid for Pregnancies, DiabetesPedigreeFunction, Age, and SourceLabel. Zero is a missing-value sentinel for Glucose (5), BloodPressure (35), SkinThickness (224), Insulin (371), and BMI (11).
- These sentinel zeros are retained during this inspection and will be handled during preprocessing rather than changed here.

## 3. Preprocessing and Fixed Data Split

### 3.1 Feature and Target Definitions

In [ ]:
# Confirm the semantic schema and create the modeling target.
COLUMN_NAMES = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "SourceLabel",
]
assert list(df.columns) == COLUMN_NAMES
assert RAW_SHAPE == (759, 9)
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == EXPECTED_SHA256
assert set(df["SourceLabel"].unique()) == {0, 1}

df["Diabetes"] = 1 - df["SourceLabel"]
assert set(df["Diabetes"].unique()) == {0, 1}
assert df["Diabetes"].value_counts().to_dict() == {0: 496, 1: 263}

FEATURE_NAMES = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
]
MISSING_ZERO_COLUMNS = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
]

X = df[FEATURE_NAMES].copy()
y = df["Diabetes"].astype("int64").copy()

print("SourceLabel distribution:")
display(df["SourceLabel"].value_counts().sort_index().rename("count"))
print("Remapped Diabetes distribution:")
display(y.value_counts().sort_index().rename("count"))

### 3.2 Missing-Value Treatment

In [ ]:
# Replace only medically invalid zero sentinels; preserve valid zeros.
X[MISSING_ZERO_COLUMNS] = X[MISSING_ZERO_COLUMNS].replace(0, np.nan)
missing_sentinel_counts = X.isna().sum()
expected_missing_sentinel_counts = pd.Series({
    "Glucose": 5,
    "BloodPressure": 35,
    "SkinThickness": 224,
    "Insulin": 371,
    "BMI": 11,
})
assert missing_sentinel_counts[MISSING_ZERO_COLUMNS].equals(expected_missing_sentinel_counts)
print("Missing-sentinel counts after replacement:")
display(missing_sentinel_counts)

### 3.3 Fixed 70/15/15 Split

In [ ]:
from sklearn.model_selection import train_test_split

# Split row indices so both frameworks use the exact same observations.
ROW_INDICES = np.arange(len(df))
train_indices, temporary_indices = train_test_split(
    ROW_INDICES,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)
validation_indices, test_indices = train_test_split(
    temporary_indices,
    test_size=0.50,
    random_state=SEED,
    stratify=y.iloc[temporary_indices],
)

assert len(train_indices) == 531
assert len(validation_indices) == 114
assert len(test_indices) == 114
assert len(set(train_indices) & set(validation_indices)) == 0
assert len(set(train_indices) & set(test_indices)) == 0
assert len(set(validation_indices) & set(test_indices)) == 0
assert len(set(train_indices) | set(validation_indices) | set(test_indices)) == len(df)
assert set(train_indices) | set(validation_indices) | set(test_indices) == set(ROW_INDICES)

X_train_raw = X.iloc[train_indices].copy()
X_val_raw = X.iloc[validation_indices].copy()
X_test_raw = X.iloc[test_indices].copy()
y_train_int = y.iloc[train_indices].copy()
y_val_int = y.iloc[validation_indices].copy()
y_test_int = y.iloc[test_indices].copy()

print("Split sizes:", {
    "train": len(train_indices),
    "validation": len(validation_indices),
    "test": len(test_indices),
})
for split_name, split_target in {
    "train": y_train_int,
    "validation": y_val_int,
    "test": y_test_int,
}.items():
    distribution = pd.DataFrame({
        "count": split_target.value_counts().sort_index(),
        "percentage": (split_target.value_counts(normalize=True).sort_index() * 100).round(4),
    })
    print(f"{split_name} class distribution:")
    display(distribution)

### 3.4 Training-Only Imputation and Scaling

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Fit both preprocessing steps on training data only to prevent leakage.
imputer = SimpleImputer(strategy="median")
X_train_imputed = imputer.fit_transform(X_train_raw)
X_val_imputed = imputer.transform(X_val_raw)
X_test_imputed = imputer.transform(X_test_raw)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_imputed).astype(np.float32)
X_val = scaler.transform(X_val_imputed).astype(np.float32)
X_test = scaler.transform(X_test_imputed).astype(np.float32)

y_train = y_train_int.to_numpy(dtype=np.float32)
y_val = y_val_int.to_numpy(dtype=np.float32)
y_test = y_test_int.to_numpy(dtype=np.float32)

assert not np.isnan(X_train).any()
assert not np.isnan(X_val).any()
assert not np.isnan(X_test).any()

print("Preprocessing complete: median imputation and StandardScaler were fit on X_train only.")

### 3.5 Preprocessing Verification

In [ ]:
def print_split_verification(name, features, target):
    counts = pd.Series(target).value_counts().sort_index()
    percentages = (pd.Series(target).value_counts(normalize=True).sort_index() * 100).round(4)
    print(f"{name} shape: X={features.shape}, y={target.shape}")
    print(f"{name} class counts: {counts.to_dict()}")
    print(f"{name} class percentages: {percentages.to_dict()}")

print("Raw dataset shape:", RAW_SHAPE)
print("Feature names:", FEATURE_NAMES)
print_split_verification("Train", X_train, y_train)
print_split_verification("Validation", X_val, y_val)
print_split_verification("Test", X_test, y_test)
print("NaN values remaining:", {
    "train": int(np.isnan(X_train).sum()),
    "validation": int(np.isnan(X_val).sum()),
    "test": int(np.isnan(X_test).sum()),
})
print("Final dtypes:", {
    "X_train": X_train.dtype,
    "X_val": X_val.dtype,
    "X_test": X_test.dtype,
    "y_train": y_train.dtype,
    "y_val": y_val.dtype,
    "y_test": y_test.dtype,
})
print("Training feature means:", np.round(X_train.mean(axis=0), 8))
print("Training feature standard deviations:", np.round(X_train.std(axis=0), 8))
print("Index overlap checks:", {
    "train_validation": len(set(train_indices) & set(validation_indices)) == 0,
    "train_test": len(set(train_indices) & set(test_indices)) == 0,
    "validation_test": len(set(validation_indices) & set(test_indices)) == 0,
    "complete_coverage": set(train_indices) | set(validation_indices) | set(test_indices) == set(ROW_INDICES),
})

The imputer and scaler are fit only on the training subset so that information from validation and test rows cannot influence the learned median or scaling parameters. This prevents data leakage and keeps the validation and test measurements honest for later model evaluation. The resulting processed arrays and saved row-index splits are shared by the PyTorch and TensorFlow experiments.

## 4. Exploratory Data Visualization

These figures use the full dataset for descriptive analysis. Medically invalid zero sentinels are treated as missing only in Glucose, BloodPressure, SkinThickness, Insulin, and BMI; valid zeros in the other columns are preserved.

### 4.1 Correlation Matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

EDA_FEATURE_NAMES = FEATURE_NAMES.copy()
eda_df = df[EDA_FEATURE_NAMES].copy()
eda_df[MISSING_ZERO_COLUMNS] = eda_df[MISSING_ZERO_COLUMNS].replace(0, np.nan)
eda_df["Diabetes"] = df["Diabetes"]

correlation_matrix = eda_df.corr(method="pearson")
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(11, 9))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"},
)
plt.title("Feature and Diabetes Correlations (Scaled Values)")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
feature_target_correlations = (
    correlation_matrix["Diabetes"]
    .drop("Diabetes")
    .sort_values(key=lambda values: values.abs(), ascending=False)
)
print("Feature-to-target correlations sorted by absolute magnitude:")
display(feature_target_correlations.rename("Pearson correlation"))

Glucose has the strongest positive relationship with the remapped Diabetes target (r = 0.491538), followed by BMI (r = 0.315051) and Insulin (r = 0.303797). Age has the weakest positive relationship (r = 0.112565), while no feature has a negative feature-to-target correlation in this descriptive analysis. Most relationships with the target are weak to moderate. These correlations describe linear association in the observed data and do not establish causation or medical effects.

### 4.2 Feature Distributions

In [ ]:
distribution_df = df[FEATURE_NAMES + ["Diabetes"]].copy()
distribution_df[MISSING_ZERO_COLUMNS] = distribution_df[MISSING_ZERO_COLUMNS].replace(0, np.nan)
class_palette = {0: "#4C78A8", 1: "#E45756"}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for index, (axis, feature) in enumerate(zip(axes.flat, FEATURE_NAMES)):
    sns.histplot(
        data=distribution_df,
        x=feature,
        hue="Diabetes",
        bins=20,
        stat="density",
        common_norm=False,
        element="step",
        fill=False,
        palette=class_palette,
        legend=False,
        ax=axis,
    )
    axis.set_title(feature)
    axis.set_xlabel("Scaled feature value")
    axis.set_ylabel("Density")

from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor=class_palette[0], edgecolor=class_palette[0], label="Diabetes = 0"),
    Patch(facecolor=class_palette[1], edgecolor=class_palette[1], label="Diabetes = 1"),
]
fig.legend(handles=legend_handles, title="Target class", loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("Feature Distributions by Diabetes Class (Scaled Values)", y=1.07)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURES_DIR / "feature_distributions.png", dpi=300, bbox_inches="tight")
plt.show()

The class distributions overlap across all eight features, although the Diabetes=1 group appears shifted toward larger scaled values for Glucose, BMI, Insulin, and Pregnancies. These visual differences are descriptive rather than tests of statistical significance. Insulin and SkinThickness have many missing zero sentinels, so their plotted distributions use fewer observations and should be interpreted with particular caution.

In [ ]:
class_counts = df["Diabetes"].value_counts().sort_index()
print("Class distribution check:")
print(f"Diabetes=0: {class_counts[0]}")
print(f"Diabetes=1: {class_counts[1]}")
print("The moderate class imbalance is why stratification was used in the fixed split.")